### Prompt Evaluation
- to build reliable AI apps, two crucial concepts are required:
    - prompt engineering: techniques to write better prompts
    - prompt evaluation: measuring how well the prompts work

Prompt evaluation measures how effective they are through automated testing
- test against expected answers
- compare different versions of the same prompt
- reviewing outputs for errors

There are 3 options after writing a prompt
1. only testing the prompt once, before shipping it, has a significant risk of it breaking in production 
2. testing the prompt multiple times, and tweaking it for one or two edge cases, users providing unexpected outputs can still break through
3. running the prompt through an evaluation pipeline to score it, then adjust the prompt based on objective metrics. This approach takes more time, work, and cost, but provides more reliability 

The third option is the best for production, since the first two methods underestimate how many edge cases real users will encounter
- when a prompt is deployed, users will interact in ways that can't be anticipated
- a good prompt with limited testing can quickly break down when there is a full variety of real-world inputs
- The third option is the more systematic approach to prompt development:
    - by running the prompt through an evaluation pipeline, there are objective metrics that display its performance across a broader range of test cases allowing you to
        - Identify weaknesses before they become production issues
        - Compare different prompt versions objectively
        - Iterate with confidence based on measurable improvements
        - Build more reliable AI applications

### A typical eval workflow
Prompt Evaluation Workflow:
- there are many different ways to assemble a workflow, there are different open source and paid options

The typical evaluation workflow has 5 key steps that improve prompts through objective measurement
1. Draft an initial prompt
    - this should be a baseline simple prompt that can be improved on
    - for this prompt evaluation example: `Please answer the user's question:`
2. Create an evaluation dataset
    - contains some number of questions that could be asked of the prompt
    - this dataset can be assembled by hand or generated by claude
    - for this prompt evaluation example the questions are the following questions in a list:
        - "What's 2+2"
        - "How do I make oatmeal"
        - "How far away is the Moon?"
    - for real world evaluations there could be up to thousands of different records in the dataset
3. Feed to Claude
    - once the dataset is assembled, the prompt and questions are fed into Claude
    - then responses from claude are returned for each question
    - for our example: "What's 2+2" becomes: `Please answer the user's question: What's 2+2?`
        - claude would return 2 + 2 = 4 for the first question
        - and the overall score is 7.66
4. Feed through a grader
    - once the answers from claude are received, the question and answers from claude are returned and fed to a grader one by one
    - the grader evaluates the quality of the answer and returns a score on a scale from 1-10 (where 10 is a perfect answer, and lower scores mean that the answer can be improved)
    - then the average of all the questions gives an objective measurement
5. Change prompt and repeat
    - after the score, the prompt can be changed and the entire pipeline can be ran again to see if the changes improve performance
    - for example: adding more guidance to the end of the prompt
        - `prompt = f"""Please answer the user's question: {question} Answer the question with ample detail"""`
        - in this example: the changes resulted in a prompt of 8.7 increasing the score
    

### Generating test datasets
- building a custom prompt evaluation workflow starts with 
    - creating an initial prompt
    - generating test data to see the performance
- each object in the dataset contains a task that will be merged into the prompt: in this case a coding task

Goal: Our prompt needs to assist users in writing three specific types of output for AWS use cases:
- Python code
- JSON configuration files
- Regular expressions
The key requirement is that when a user requests help with a task, we return clean output in one of these formats without any extra explanations, headers, or footers

With a starting prompt of:
- prompt = f"""
Please provide a solution to the following task:
{task}
"""

Creating an Evaluation Dataset
- contains inputs that we'll feed into our prompt
- the combination of prompts and inputs will be ran and evaluated
- the dataset will be an array of JSON objects, where each object contains a task property that describes a task for claude to accomplish
- this dataset can be written by hand or automatically generated by claude. A faster model like haiku can be used, since the task is generating test data

In [112]:
from dotenv import load_dotenv

load_dotenv()

from anthropic import Anthropic
client = Anthropic()
model = "claude-haiku-4-5"

In [113]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [64]:
import json

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
# prompt to generate the dataset of questions includes
    # tasks, output type, example output, additional instructions, number of objects generated

    messages = []
    add_user_message(messages, prompt) 
    add_assistant_message(messages, "```json") # since we are asking for a json output, Assistant Message Prefilling will include the markdown starting for json
    text = chat(messages, stop_sequences=["```"]) # stop once ``` is going to be generated to close the markdown file
    
    return json.loads(text)

In [65]:
dataset = generate_dataset()

dataset
# the dataset now contains all the questions needed

[{'task': "Write a Python function that extracts the AWS region from an S3 bucket URL. The function should take a URL like 's3://my-bucket.s3.us-west-2.amazonaws.com/key' and return 'us-west-2'."},
 {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific S3 bucket named 'my-data-bucket'. The policy should include the appropriate AWS principals and actions."},
 {'task': "Write a regular expression that matches and validates AWS EC2 instance IDs. Valid instance IDs follow the format 'i-' followed by exactly 17 hexadecimal characters (e.g., 'i-0abcd1234efgh5678')."}]

### Running the eval

Building the core functions: taking each test case, merging it with our prompt, feeding it to Claude, and then grading the results
1. Merging prompt and test case inputs
    - takes a test case and merges it with the prompt template
2. Running test cases and grading them
    - orchestrates the running of a single test case and grading the result
3. Running the whole evaluation pipeline (orchestrator)
    - coordinates the entire evaluation process:

This pipeline is the foundation of most AI evaluation systems:
- complexity in industry grade pipeline includes
    - better prompts
    - sophisticated grading
    - performance optimizations (rust/c++)

In [114]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
""" 
    
    # sending the test cases to claude
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [67]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case) # this calls run_prompt, so run_prompt is not needed to be included in run_eval
    
    # TODO - Grading
    score = 10 # hard coding grader for now, will use actual methods for grading in later lesson
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    } # returns a set of what the input and output was as well as the score

In [115]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = [] # list to store the test cases
    
    for test_case in dataset: # for each result in the dataset
        result = run_test_case(test_case) # merging the prompt and test case
        results.append(result) # adding the results into the list 
    
    return results

In [69]:
# running the evaluation pipeline

results = run_eval(dataset)

In [70]:
print(json.dumps(results, indent=2)) # the output has a lot of additional comments and headers so formatting instructions will have to be included
# each result has: output(response from Claude), test_case(original test case), score (hardcoded evaluation score)
# the core evaluation pipeline has been built, the only thing missing is the grading system

[
  {
    "output": "# AWS S3 Region Extractor\n\nHere's a comprehensive solution with multiple approaches:\n\n```python\nimport re\nfrom urllib.parse import urlparse\n\ndef extract_aws_region_from_s3_url(url: str) -> str:\n    \"\"\"\n    Extract AWS region from an S3 bucket URL.\n    \n    Supports multiple S3 URL formats:\n    - s3://my-bucket.s3.us-west-2.amazonaws.com/key\n    - https://my-bucket.s3.us-west-2.amazonaws.com/key\n    - https://s3.us-west-2.amazonaws.com/my-bucket/key\n    - s3://bucket-name/key (returns 'us-east-1' as default)\n    \n    Args:\n        url: S3 bucket URL\n        \n    Returns:\n        AWS region string (e.g., 'us-west-2')\n        \n    Raises:\n        ValueError: If URL format is invalid or region cannot be extracted\n    \"\"\"\n    if not url:\n        raise ValueError(\"URL cannot be empty\")\n    \n    # Pattern 1: Virtual-hosted-style URL (bucket in subdomain)\n    # s3://my-bucket.s3.us-west-2.amazonaws.com/key\n    pattern1 = r'\\.s3[.-](

### Model based grading
- provides objective signals about the quality of output: true/false or a scale from 1-10
- There are three main ways to grade model outputs
    - Code graders - Programmatically evaluate outputs using checks (has to return a usable signal: score of 1-10)
        - checking output length
        - verifying whether outputs includes certain words or not
        - syntax validation for Python, JSON, etc
        - reliability scores
    - Model graders - Use another AI model to assess the quality (make another API call)
        - response quality
        - quality of instruction following
        - completeness
        - safety
    - Human graders - Have people manually review and score outputs (slow and tedious)
        - general response quality
        - comprehensiveness
        - depth
        - conciseness
        - relevance
- Evaluation criteria
    - format
        - returns only python, JSON, Regex **without explanation**
    - valid syntax
        - produced code should have valid syntax
    - task following
        - response should directly and clearly answer the user's question
        - generated code should be accurate
- the first two criteria fall under *code graders* while the last criteria falls under *model grader*

In [116]:
def grade_by_model(test_case, output): # test case an the returned output from claude
    # evaluation prompt including:
        # task def
        # task and solution injection using {}
        # instructions (on what metrics and methods to use when grading the model, and reasoning to find out the reason for weak or strong scores for improvement)
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [117]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case) # this calls run_prompt, so run_prompt is not needed to be included in run_eval
    
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    # TODO: could extract strengths and weaknesses as well
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    } # returns a set of what the input and output was as well as the score

In [ ]:
results = run_eval(dataset)
results

# this includes actual calculated scores and the reasoning for each of the scores

[{'output': '# AWS S3 Region Extraction Function\n\nHere\'s a comprehensive solution with multiple approaches:\n\n```python\nimport re\nfrom urllib.parse import urlparse\n\ndef extract_s3_region(url: str) -> str:\n    """\n    Extracts the AWS region from an S3 bucket URL.\n    \n    Supports multiple S3 URL formats:\n    - Virtual-hosted style: s3://my-bucket.s3.us-west-2.amazonaws.com/key\n    - Path style: s3://s3.us-west-2.amazonaws.com/my-bucket/key\n    - S3 URI: s3://my-bucket/key (returns \'us-east-1\' as default)\n    \n    Args:\n        url: S3 bucket URL string\n        \n    Returns:\n        AWS region code (e.g., \'us-west-2\')\n        \n    Raises:\n        ValueError: If region cannot be extracted from URL\n    """\n    # Pattern to match region in S3 URLs\n    # Matches: .s3.<region>.amazonaws.com or s3.<region>.amazonaws.com\n    pattern = r\'s3[.-]([a-z0-9\\-]+)\\.amazonaws\\.com\'\n    \n    match = re.search(pattern, url)\n    \n    if match:\n        return matc

In [118]:
from statistics import mean

def run_eval(dataset):
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}") # to get the average scores
    
    return results

In [ ]:
results = run_eval(dataset)
# running the evaluation pipeline again with the modified run_eval function to get the average scores

Average score: 6.333333333333333


### Code based grading
- verifies that the generated code has valid syntax and follows the correct format

Checklist Process:
- add functions to validate JSON/Python/Regex
- make sure the dataset test cases includes the type of generated content (if the code generated is JSON/Python/Regex, for knowledge of which test to apply)
- update draft prompt to make it clear only the relevant language is wanted
- merge the scores from the model grader and the code grader

In [119]:
# functions to validate JSON/Python/Regex

import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_syntax(response, test_case):
    format = test_case["format"]
    
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [96]:
# adding dataset testcases to include the language type

import json

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
# prompt to generate the dataset of questions includes
    # tasks, output type, example output, additional instructions, number of objects generated

    messages = []
    add_user_message(messages, prompt) 
    add_assistant_message(messages, "```json") # since we are asking for a json output, Assistant Message Prefilling will include the markdown starting for json
    text = chat(messages, stop_sequences=["```"]) # stop once ``` is going to be generated to close the markdown file
    
    return json.loads(text)

In [97]:
dataset = generate_dataset()
dataset

[{'task': "Extract the AWS region from an S3 bucket ARN like 'arn:aws:s3:::my-bucket-us-east-1'",
  'format': 'regex'},
 {'task': 'Parse an AWS CloudFormation template and return a JSON object containing only the resources section',
  'format': 'json'},
 {'task': 'Write a Python function that converts an AWS IAM policy ARN to its account ID',
  'format': 'python'}]

In [120]:
# update draft prompt that we only want generation of JSON/Python/Regex and no other values

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or plain Regex
* Do not add any comments or commentary or explanation
""" 
    
    # sending the test cases to claude
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code") # since at this point it is unknown whether its going to be Python, JSON, or plain Regex, so instead just putting code is enough to get it started, without specifying
    output = chat(messages, stop_sequences=["```"])
    return output

In [121]:
# merging the coding and model scores

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case) # this calls run_prompt, so run_prompt is not needed to be included in run_eval
    
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    # TODO: could extract strengths and weaknesses as well
    
    # code grader
    syntax_score = grade_syntax(output, test_case)
    
    # merging using average between model and code graders
    score = (model_score + syntax_score) / 2
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    } # returns a set of what the input and output was as well as the score

In [122]:
from statistics import mean

def run_eval(dataset):
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}") # to get the average scores
    
    return results

In [124]:
results = run_eval(dataset)

Average score: 6.333333333333333


In [102]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\n\ndef extract_region_from_s3_arn(arn):\n    match = re.search(r'(us|eu|ap|ca|sa|me|af)-(east|west|south|north|central|northeast|southeast)-\\d', arn)\n    if match:\n        return match.group(0)\n    return None\n\n# Test\nprint(extract_region_from_s3_arn('arn:aws:s3:::my-bucket-us-east-1'))\n",
    "test_case": {
      "task": "Extract the AWS region from an S3 bucket ARN like 'arn:aws:s3:::my-bucket-us-east-1'",
      "format": "regex"
    },
    "score": 6.5,
    "reasoning": "While the solution demonstrates regex competency, it conflates bucket naming conventions with actual ARN structure. AWS S3 bucket ARNs (arn:aws:s3:::bucket-name) inherently lack region information\u2014regions must be inferred from bucket names or metadata, not parsed from the ARN itself. The regex pattern, though reasonable for extracting region-like strings, is both too restrictive (missing valid regions) and applied to the wrong semantic layer (bucket name vs. ARN format).

### Quiz on prompt evaluation
- giving the model grader more context on what a good solution looks like, by adding a prompt in both the dataset generation (an example of what a output looks like) and a prompt in the model grader (so that the model knows what to look for when grading questions)
- this process focuses more on prompt engineering, which what the next chapter covers

In [125]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [126]:
dataset = generate_dataset()
dataset

[{'task': "Extract AWS S3 bucket names from CloudFormation template resource names following the pattern 'my-bucket-{environment}-{region}'",
  'format': 'regex',
  'solution_criteria': "Regex should correctly match bucket names with format 'my-bucket-' followed by alphanumeric characters and hyphens, capturing the environment and region segments"},
 {'task': 'Parse AWS CloudWatch log event and extract the timestamp, log level, and message into a Python dictionary',
  'format': 'python',
  'solution_criteria': "Function should accept a CloudWatch log line string and return a dictionary with keys 'timestamp', 'level', and 'message' with correct values extracted"},
 {'task': 'Create a JSON configuration object for an AWS Lambda function that includes environment variables for database credentials, timeout settings, and memory allocation',
  'format': 'json',
  'solution_criteria': 'JSON should include proper Lambda configuration structure with Environment.Variables, Timeout, and MemorySi

In [127]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """ # informing the model grader what a good solution looks like (Criteria) by grabbing the solution criteria from the test case that was generated in the testing dataset

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [132]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [133]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [134]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [ ]:
results = run_eval(dataset)
# with the additional tweaking and improvements of the prompt and the generation of the dataset, the average score has gone up significantly

Average score: 7.666666666666667


In [136]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\nimport json\nimport sys\n\ndef extract_s3_buckets(template_str):\n    pattern = r'my-bucket-\\w+-\\w+'\n    matches = re.findall(pattern, template_str)\n    return matches\n\nif __name__ == \"__main__\":\n    template = sys.stdin.read()\n    buckets = extract_s3_buckets(template)\n    print(json.dumps(buckets))\n",
    "test_case": {
      "task": "Extract AWS S3 bucket names from CloudFormation template resource names following the pattern 'my-bucket-{environment}-{region}'",
      "format": "regex",
      "solution_criteria": "Regex should correctly match bucket names with format 'my-bucket-' followed by alphanumeric characters and hyphens, capturing the environment and region segments"
    },
    "score": 7.0,
    "reasoning": "The solution addresses the core requirement but has a critical regex flaw. The pattern `my-bucket-\\w+-\\w+` will not correctly match bucket names containing hyphens within the environment or region segments (e.g., 'my-bucket